In [5]:
!python --version

Python 3.11.14


In [6]:
import os, sys
import pandas as pd
import requests
import time

from pathlib import Path

ROOT = Path().resolve().parent.parent
ROOT_SCR = os.path.join(ROOT, "src")

if str(ROOT_SCR) not in sys.path:
    sys.path.append(str(ROOT_SCR))

print("ROOT:", ROOT)
print("ROOT_SCR added:", ROOT_SCR)

from libs.calc_degs_lib import CALC_DEGS
from libs.tcga_gdc_lib import *
from libs.Basic import *


ROOT: /home/flavio/uv
ROOT_SCR added: /home/flavio/uv/src


### Defaults

In [7]:
ROOT_DATA = '../data'
ROOT_DATA = Path(ROOT_DATA)

gdc = GDC(ROOT_DATA0=ROOT_DATA)

os.listdir(ROOT_DATA)[:10]


['cancer', 'reactome', 'vector_store', 'TCGA', 'GTEx', 'gdc_programs.txt']

In [8]:
os.listdir(gdc.root_data)[:10]

['app_main_local.py',
 'gtex_normal_tissue.ipynb',
 'docker_config.ipynb',
 'gtex_normal_tissue_dvlp.ipynb',
 'tcga_gdc_run_R_expression_dvlp.ipynb']

### Get all programs

In [9]:
force=False
verbose=True

prog_list = gdc.get_gdc_progams(force=force, verbose=verbose)

np.array(prog_list)


File read at '../data/gdc_programs.txt'


array(['TCGA', 'MATCH', 'TARGET', 'CGCI', 'CMI', 'APOLLO', 'BEATAML1.0',
       'CPTAC', 'MP2PRT', 'ALCHEMIST', 'CCDI', 'CCG', 'CDDP_EAGLE',
       'CTSP', 'EXCEPTIONAL_RESPONDERS', 'FM', 'HCMI', 'MMRF', 'NCICCR',
       'OHSU', 'ORGANOID', 'RC', 'REBC', 'TRIO', 'VAREPOP', 'WCDT'],
      dtype='<U22')

### Primary sites given a program

In [10]:
gdc.url_gdc_project

'https://api.self.cancer.gov/projects'

In [11]:
prog_id = 'TCGA'

df_psi = gdc.get_primary_sites(prog_id=prog_id, force=False, verbose=verbose)
print(len(df_psi))

df_psi.head(3)

Table opened ((33, 5)) at '../data/TCGA/primary_site_program_TCGA.tsv'
33


,psi_id,primary_site,project_id,disease_type,name
0,TCGA-ACC,Adrenal gland,TCGA-ACC,Adenomas and Adenocarcinomas,Adrenocortical Carcinoma
1,TCGA-PCPG,"Adrenal gland, Retroperitoneum and peritoneum,...",TCGA-PCPG,Paragangliomas and Glomus Tumors,Pheochromocytoma and Paraganglioma
2,TCGA-BLCA,Bladder,TCGA-BLCA,"Epithelial Neoplasms, NOS, Squamous Cell Neopl...",Bladder Urothelial Carcinoma


### GTEx

In [12]:
df_gtex = gdc.read_GTEx_to_TCGA_table()
print(len(df_gtex))

df_gtex.head(3)

33


,tcga_project_id,tcga_primary_site,gtex_dataset_id,preferred_gtex_id,gtex_tissue_site_detail_ids,mapping_confidence,notes,source_gtex_api
0,TCGA-ACC,Adrenal Gland,gtex_v8,Adrenal_Gland,Adrenal_Gland,high,Adrenocortical carcinoma; direct adrenal gland...,https://gtexportal.org/api/v2/redoc
1,TCGA-BLCA,Bladder,gtex_v8,Bladder,Bladder,high,"Direct bladder GTEx tissue, but GTEx bladder s...",https://gtexportal.org/api/v2/redoc
2,TCGA-BRCA,Breast,gtex_v8,Breast_Mammary_Tissue,Breast_Mammary_Tissue,high,Breast carcinoma; GTEx mammary/breast tissue.,https://gtexportal.org/api/v2/redoc


In [13]:
verbose = True

psi_id = 'TCGA-BRCA'
psi_id = 'TCGA-LUAD'

gtex_id, gtex_tissue_ids = gdc.find_GTEx_to_TCGA_row(psi_id=psi_id, verbose=verbose)
print(">>>", gtex_id, gtex_tissue_ids)


Found 'Lung' tissue 'Lung'
>>> Lung Lung


### Get df_tumor - entropy - to find homogeneous rows

In [14]:
verbose=False

df_tumor, df_normal = gdc.get_file_expression_both_tumor_and_normal(psi_id=psi_id, verbose=verbose)

len(df_tumor), len(df_normal)

Dowloading normal files: 0.........10.........20.........30.........40.........50.........60.........70.........80.........90.........
Dowloading tumor files: 0.........10.........20.........30.........40.........50.........60.........70.........80.........90.........100........


(240796, 240796)

In [15]:
df_tumor.head(3)

,gene_id,symbol,gene_type,tumor_1,tumor_2,tumor_3,tumor_4,tumor_5,tumor_6,tumor_7,tumor_8,tumor_9,tumor_10,tumor_11,tumor_12
0,ENSG00000000003,TSPAN6,protein_coding,1604,2392,1373,803,799,5095,0,720,2,792,1489,3870
1,ENSG00000000005,TNMD,protein_coding,0,0,8,2,1,0,0,0,14,0,0,1
2,ENSG00000000419,DPM1,protein_coding,459,816,713,799,400,3372,3,765,39,1043,345,1380


In [16]:
df_tumor2 = gdc.add_entropy(df_tumor)
print(len(df_tumor2))
df_tumor2.head(3)


14276


,gene_id,symbol,gene_type,tumor_1,tumor_2,tumor_3,tumor_4,tumor_5,tumor_6,tumor_7,tumor_8,tumor_9,tumor_10,tumor_11,tumor_12,total_sum,entropy,entropy_q
0,ENSG00000000003,TSPAN6,protein_coding,1604,2392,1373,803,799,5095,0,720,2,792,1489,3870,18939,2.977549,4
2,ENSG00000000419,DPM1,protein_coding,459,816,713,799,400,3372,3,765,39,1043,345,1380,10134,2.970012,4
3,ENSG00000000457,SCYL3,protein_coding,542,432,557,699,264,995,30,674,168,352,283,1706,6702,3.141205,7


In [17]:
tumor_cols = [c for c in df_tumor.columns if c.startswith("tumor_")]

df_tumor3 = gdc.select_top_entropy(df_tumor2, q=0.1)
print(len(df_tumor3))
df_tumor3.head(3)

1428


,gene_id,symbol,gene_type,tumor_1,tumor_2,tumor_3,tumor_4,tumor_5,tumor_6,tumor_7,tumor_8,tumor_9,tumor_10,tumor_11,tumor_12,total_sum,entropy,entropy_q
0,ENSG00000135919,SERPINE2,protein_coding,189,821,1026,2040,1170,216535,0,119,5,599,193,1394,224091,0.306282,0
1,ENSG00000104267,CA2,protein_coding,187,209,400,91,296,394,12,57687,4,125,89,549,60043,0.348895,0
2,ENSG00000096088,PGC,protein_coding,632,757,1394,10,621,2151,16,159907,4,20,182,2587,168281,0.407563,0


In [18]:
geneid_list = list(np.unique(df_tumor3.gene_id))
len(geneid_list)

1037

### GTEx portal - data manually downloaded

In [19]:
print(len(geneid_list))
np.array(geneid_list[:50])

1037


array(['ENSG00000000460', 'ENSG00000001626', 'ENSG00000003989',
       'ENSG00000004468', 'ENSG00000005020', 'ENSG00000006016',
       'ENSG00000006210', 'ENSG00000006534', 'ENSG00000006747',
       'ENSG00000007402', 'ENSG00000008735', 'ENSG00000008853',
       'ENSG00000012171', 'ENSG00000012223', 'ENSG00000017483',
       'ENSG00000018236', 'ENSG00000019102', 'ENSG00000019186',
       'ENSG00000019582', 'ENSG00000021300', 'ENSG00000022267',
       'ENSG00000023171', 'ENSG00000025423', 'ENSG00000025708',
       'ENSG00000026103', 'ENSG00000026508', 'ENSG00000029153',
       'ENSG00000029993', 'ENSG00000034053', 'ENSG00000038002',
       'ENSG00000042980', 'ENSG00000047457', 'ENSG00000047597',
       'ENSG00000047662', 'ENSG00000047936', 'ENSG00000048052',
       'ENSG00000051128', 'ENSG00000053747', 'ENSG00000053918',
       'ENSG00000057294', 'ENSG00000059728', 'ENSG00000060718',
       'ENSG00000060982', 'ENSG00000062038', 'ENSG00000064102',
       'ENSG00000064225', 'ENSG000000647

In [20]:
gdc.gtex_id

'Lung'

In [21]:
verbose=False
force=False

want_run = False

if want_run:
    df_gtex = gdc.get_gtex_TPM_expression_for_geneid_list(
                    geneid_list=geneid_list,
                    datasetId="gtex_v8", page_size=1000,
                    sleep = 0.1, timeout=60,
                    force=force, verbose=verbose)
else:
    df_gtex = pd.DataFrame()

print(df_gtex.shape)

(0, 0)


### GTEx download

https://www.gtexportal.org/home/downloads/adult-gtex/overview

In [22]:
url_counts = (
    "https://storage.googleapis.com/adult-gtex/bulk-gex/v11/rna-seq/"
    "GTEx_Analysis_2025-08-22_v11_RNASeQCv2.4.3_gene_reads.gct.gz"
)

url_tpm = (
    "https://storage.googleapis.com/adult-gtex/bulk-gex/v11/rna-seq/"
    "GTEx_Analysis_2025-08-22_v11_RNASeQCv2.4.3_gene_tpm.gct.gz"
)

url_meta = (
    "https://storage.googleapis.com/adult-gtex/metadata/"
    "GTEx_Analysis_v11_Annotations_SampleAttributesDS.txt"
)


filename_counts = gdc.root_gtex / gdc.fname_GTEx_counts
filename_meta   = gdc.root_gtex / gdc.fname_GTEx_meta
filename_pheno = gdc.root_gtex / gdc.fname_GTEx_pheno

# --> download manually at https://www.gtexportal.org/home/downloads/adult-gtex/bulk_tissue_expression
#                      and https://www.gtexportal.org/home/downloads/adult-gtex/metadata

# gdc.download_file(url_counts, filename_counts)
# gdc.download_file(url_meta, filename_meta)

### Phenotype - DTHHRDY (Hardy Scale of Death)

> It indicates how the donor died and how fast, which affects tissue quality.  


| Value	| Meaning  |
|---------|-----------|
| 0	| Ventilator case (on life support before death) |
| 1	| Violent and fast death (<10 min) |
| 2	| Fast death (10 min – 1 hour) |
| 3	| Intermediate death (1 – 24 hours) |
| 4	| Slow death (>24 hours, prolonged illness) |
| NA	| Unknown |


In [23]:
df_pheno = gdc.read_GTEx_pheno(verbose=verbose)

print(len(df_pheno))
df_pheno.head(3)

981


,SUBJID,SEX,AGE,DTHHRDY
0,GTEX-1117F,2,60-69,4.0
1,GTEX-111CU,1,50-59,0.0
2,GTEX-111FC,1,60-69,1.0


In [24]:
df_meta = gdc.read_GTEx_metadata(verbose=verbose)

print(len(df_meta))
df_meta.head(3)

48231


,SAMPID,SMATSSCR,SMCENTER,SMPTHNTS,SMRIN,SMTS,SMTSD,SMUBRID,SMTSISCH,SMTSPAX,...,SMSHRTRT,SMSMRDHQ,SMSMRTHQ,SMPRERDHQ,SMPRERTHQ,SMSMGNDT,SMPREGNDT,SMRDLNMN,SMRDLNMD,SMRDLNSD
0,BMS-X4LF-0126-SM-4JBHL,NaN,B1,NaN,7.5,Thyroid,Thyroid,UBERON:0002046,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,BMS-X4LF-0226-SM-4JBJ3,NaN,B1,NaN,6.9,Blood Vessel,Artery - Pulmonary,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,BMS-X4LF-0326-SM-4JBIR,NaN,B1,NaN,7.4,Muscle,Muscle - Skeletal,UBERON:0011907,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


### Healthy donors

1. Filter the tissue
2. Filter for high quality (Hardy Scale 1 or 2 are 'fast' deaths, less stress)
  . DTHHRDY: 1 = Ventilator, 2 = Fast death of natural causes
3. Sort by RIN score (SMRIN) to get the best preserved RNA

In [25]:
df_meta_ctrl = gdc.prepare_gtex_df_control()

print(df_meta_ctrl.shape)
df_meta_ctrl.head(5)

(547, 123)


,SAMPID,SMATSSCR,SMCENTER,SMPTHNTS,SMRIN,SMTS,SMTSD,SMUBRID,SMTSISCH,SMTSPAX,...,SMPRERTHQ,SMSMGNDT,SMPREGNDT,SMRDLNMN,SMRDLNMD,SMRDLNSD,SUBJID,SEX,AGE,DTHHRDY
0,GTEX-1A32A-0726-SM-F4BEK,1.0,C1,2 pieces; emphysema; focal vascular congestion...,8.7,Lung,Lung,UBERON:0008952,697.0,851.0,...,0.005925,5674.0,805.0,25.0828,22.0,6.3502,GTEX-1A32A,2.0,50-59,2.0
1,GTEX-N7MS-0926-SM-2AXU3,2.0,C1,OK for analysis,8.7,Lung,Lung,UBERON:0008952,1232.0,1561.0,...,NaN,NaN,NaN,NaN,NaN,NaN,GTEX-N7MS,1.0,60-69,2.0
2,GTEX-N7MS-0926-SM-2HMIZ,2.0,C1,OK for analysis,8.7,Lung,Lung,UBERON:0008952,1232.0,1561.0,...,NaN,NaN,NaN,NaN,NaN,NaN,GTEX-N7MS,1.0,60-69,2.0
3,GTEX-1A32A-0726-SM-731D4,1.0,C1,2 pieces; emphysema; focal vascular congestion...,8.7,Lung,Lung,UBERON:0008952,697.0,851.0,...,NaN,NaN,NaN,NaN,NaN,NaN,GTEX-1A32A,2.0,50-59,2.0
4,GTEX-14PJM-0926-SM-6AJ9Y,1.0,C1,2 pieces; moderate emphysema; mild patchy alve...,8.6,Lung,Lung,UBERON:0008952,564.0,826.0,...,NaN,NaN,NaN,NaN,NaN,NaN,GTEX-14PJM,2.0,50-59,2.0


In [26]:
verbose=False

psi_id='TCGA-LUSC'
psi_id='TCGA-BRCA'
psi_id='TCGA-LUAD'

df_tumor, df_normal = gdc.get_file_expression_both_tumor_and_normal(psi_id=psi_id, verbose=verbose)
len(df_tumor), len(df_normal)

Dowloading normal files: 0.........10.........20.........30.........40.........50.........60.........70.........80.........90.........
Dowloading tumor files: 0.........10.........20.........30.........40.........50.........60.........70.........80.........90.........100........


(240796, 240796)

In [27]:
df_tumor.head(3)

,gene_id,symbol,gene_type,tumor_1,tumor_2,tumor_3,tumor_4,tumor_5,tumor_6,tumor_7,tumor_8,tumor_9,tumor_10,tumor_11,tumor_12
0,ENSG00000000003,TSPAN6,protein_coding,1604,2392,1373,803,799,5095,0,720,2,792,1489,3870
1,ENSG00000000005,TNMD,protein_coding,0,0,8,2,1,0,0,0,14,0,0,1
2,ENSG00000000419,DPM1,protein_coding,459,816,713,799,400,3372,3,765,39,1043,345,1380


In [28]:
df_normal.head(3)

,gene_id,symbol,gene_type,normal_1,normal_2,normal_3,normal_4,normal_5,normal_6,normal_7,normal_8,normal_9,normal_10,normal_11,normal_12
0,ENSG00000000003,TSPAN6,protein_coding,303,873,396,662,280,419,1024,266,362,428,1222,619
1,ENSG00000000005,TNMD,protein_coding,0,2,1,0,1,5,7,1,0,0,2,51
2,ENSG00000000419,DPM1,protein_coding,477,481,450,439,482,421,1034,344,357,298,793,525


In [29]:
force=False
verbose=False

df_gtex_ctrl, df_meta_ctrl = gdc.calc_gtex_control(psi_id=psi_id, Nsamples=15, force=force, verbose=verbose)
print(df_gtex_ctrl.shape)

df_gtex_ctrl.head(3)

(74628, 17)


,ensemblid,symbol,GTEX-1A32A-0726-SM-731D4,GTEX-14PJM-0926-SM-6AJ9Y,GTEX-11EQ9-0226-SM-5A5JX,GTEX-WRHU-0226-SM-3MJFV,GTEX-1JMLX-0926-SM-CNNPQ,GTEX-13OW6-0826-SM-5L3GA,GTEX-YFC4-1126-SM-5RQJN,GTEX-QXCU-0626-SM-2TC69,GTEX-Q2AG-1026-SM-33HBW,GTEX-145MF-0726-SM-5Q5BT,GTEX-WZTO-0426-SM-3NM99,GTEX-13N2G-0826-SM-5IJE6,GTEX-UPJH-0826-SM-4WKFD,GTEX-1A8FM-0826-SM-793D5,GTEX-S33H-0626-SM-2XCBJ
0,ENSG00000290825.2,DDX11L16,3,0,0,0,0,0,0,1,1,0,0,1,1,1,0
1,ENSG00000223972.6,DDX11L1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
2,ENSG00000310526.1,WASH7P,207,174,210,376,507,189,110,211,559,174,342,328,388,152,451


In [ ]:

verbose=False

run_loop=False

if run_loop:
    for i, row in df_psi.iterrows():
        psi_id = row.psi_id
        primary_site = row.primary_site
        print(f">>> {psi_id} - {primary_site[:15]}", end=" ")

        df_tumor, df_normal = gdc.get_file_expression_both_tumor_and_normal(psi_id=psi_id, verbose=verbose)

        if df_tumor.empty:
            print(f"No tumor expression data found")
            continue

        df_gtex_ctrl, df_meta_ctrl = gdc.calc_gtex_control(psi_id=psi_id, Nsamples=15, force=force, verbose=verbose)

        print(f"-> Tumor: {df_tumor.shape[0]}, Normal: {df_normal.shape[0]}, GTEx: {df_gtex_ctrl.shape[0]}")


>>> TCGA-ACC - Adrenal gland No tumor expression data found
>>> TCGA-PCPG - Adrenal gland,  No tumor expression data found
>>> TCGA-BLCA - Bladder Dowloading normal files: 0.........10.
Dowloading tumor files: 0.........10.
No tumor expression data found
>>> TCGA-LGG - Brain No tumor expression data found
>>> TCGA-GBM - Brain No tumor expression data found
>>> TCGA-BRCA - Breast Dowloading normal files: 0.
Dowloading tumor files: 0.
-> Tumor: 60748, Normal: 60748, GTEx: 74628
>>> TCGA-LUAD - Bronchus and lu Dowloading normal files: 0.........10.........20.........30.........40.........50.........60.........70.........80.........90.........
Dowloading tumor files: 0.........10.........20.........30.........40.........50.........60.........70.........80.........90.........100........
-> Tumor: 240796, Normal: 240796, GTEx: 74628
>>> TCGA-LUSC - Bronchus and lu Dowloading normal files: 0.........10.........20.........30.........40.........50....
Dowloading tumor files: 0.........10.......

In [34]:
force=True
verbose=False

psi_id='TCGA-LUSC'
psi_id='TCGA-BRCA'
psi_id='TCGA-LUAD'

df_degs, df_lfc, degs_txt, msg = gdc.calc_degs(psi_id=psi_id, root_src=ROOT_SCR, lfc_cutoff=1.0, fdr_cutoff=0.05, method="edger", verbose=verbose, force=force)
    


Dowloading normal files: 0.........10.........20.........30.........40.........50.........60.........70.........80.........90.........
Dowloading tumor files: 0.........10.........20.........30.........40.........50.........60.........70.........80.........90.........100........


TypeError: CALC_DEGS.__init__() got an unexpected keyword argument 'src_dir'